# **Fine tuning de asistente legal usando Lora**

**Tópicos Especiales y Aplicaciones en IA** — Universidad EAFIT — Módulo 1 — Transformers

Integrantes:
- Martin Valencia
- Pablo Cabrejos
- Samuel Lopez
- Miguel Ortiz
---


## 0. Setup


In [1]:
# Instalamos lo necesario para LoRA, datasets y métricas.
%pip install -q -U peft datasets evaluate accelerate scikit-learn
%pip uninstall -y torchao
print('\nListo.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 10.9 MB/s eta 0:00:00
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0

Listo.


- Se importan dependencias necesarias, se fija una semilla y se usa la GPU de collab:

In [2]:
import random
import re
import unicodedata

import numpy as np
import pandas as pd
import torch
import transformers
import peft

SEED = 42

def fijar_semilla(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    transformers.set_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

fijar_semilla()

device = "cuda" if torch.cuda.is_available() else "cpu"
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("torch:", torch.__version__)
print("device:", device)

if device == "cpu":
    print(" Activa la GPU T4 en Entorno de ejecución > Cambiar tipo de entorno.")

transformers: 5.13.1
peft: 0.20.0
torch: 2.11.0+cu128
device: cuda


## 1. Cargar, unir y limpiar datos

Vamos a unir **dasatet_cross_encoder.csv** con **diccionario_articulos.csv** para así lograr que las consultas queden relacionadas con las descripciones de los articulos

In [3]:
# Traemos los datos directamente del repositorio:

REPO = "Bosnape/cabrejos-ortiz-valencia-lopez"
REF = "main"  # Al entregar, conviene reemplazarlo por el commit final del equipo.
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/{REF}/data"

pares = pd.read_csv(f"{BASE_URL}/dataset_cross_encoder.csv")
diccionario = pd.read_csv(f"{BASE_URL}/diccionario_articulos.csv")

print("Pares originales:", len(pares))
display(pares.head(3))
print()
display(diccionario.head(3))

Pares originales: 561


,consulta,articulo,tipo,label,sentencia_origen
0,Trabajé como operador de bus articulado desde ...,CST Art. 127,positivo,1,SL-3630/22
1,Trabajé como operador de bus articulado desde ...,CST Art. 128,positivo,1,SL-3630/22
2,Trabajé como operador de bus articulado desde ...,CST Art. 236,negativo_facil,0,SL-3630/22


,fuente,numero,texto_completo,n_citas_en_dataset,url_fuente
0,CGP,167,Artículo 167. Carga de la prueba. Incumbe a la...,1,https://www.funcionpublica.gov.co/eva/gestorno...
1,CGP,244,Artículo 244. Documento auténtico. Es auténtic...,1,https://www.funcionpublica.gov.co/eva/gestorno...
2,CPTSS,50,ARTICULO 50. -Extra y ultra petita. El juez (d...,1,https://www.funcionpublica.gov.co/eva/gestorno...


Ahora separamos la cita, por ejemplo CST Art. 127, en la llave (CST, 127) y hacemos el join con el texto completo del artículo.

In [4]:
def normalizar_llave(valor):
    """Normaliza tildes, mayúsculas y espacios para hacer joins robustos."""
    texto = unicodedata.normalize("NFKD", str(valor))
    texto = texto.encode("ascii", "ignore").decode("ascii")
    texto = re.sub(r"\s+", " ", texto).strip().upper()
    return texto

# Variantes de escritura de la misma fuente vistas en las citas del LLM.
ALIASES_FUENTE = {
    "CODIGO SUSTANTIVO DEL TRABAJO": "CST",
    "CODIGO SUSTANTIVO DE TRABAJO": "CST",
    "C S T": "CST",
    "CONSTITUCION POLITICA": "CP",
}

def normalizar_fuente(valor):
    fuente = normalizar_llave(valor)

    if fuente.startswith("CODIGO SUSTANTIVO DEL TRABAJO"):
        return "CST"

    if fuente.startswith("CODIGO SUSTANTIVO DE TRABAJO"):
        return "CST"

    if fuente.startswith("CONSTITUCION POLITICA"):
        return "CP"

    if fuente == "C S T":
        return "CST"

    return ALIASES_FUENTE.get(fuente, fuente)

# Misma lógica que normalizar_articulos_citados() en ds_diccionario_articulos.ipynb.
def parsear_articulo(articulo):
    texto = str(articulo).strip()

    # Ej.: CST Art. 127 / Ley 6 de 1945, Art. 1
    match = re.match(
        r"^(?P<fuente>.+?)(?:,)?\s+"
        r"Art(?:[íi]culo|\.)?\s*"
        r"(?P<numero>[0-9]+[A-Za-z]?)",
        texto,
        flags=re.IGNORECASE,
    )
    if match:
        return (
            normalizar_fuente(match.group("fuente")),
            normalizar_llave(match.group("numero")),
        )

    # Ej.: Artículo 57 numeral 5 del Código Sustantivo del Trabajo
    match = re.match(
        r"^Art(?:[íi]culo|\.)?\s*"
        r"(?P<numero>[0-9]+[A-Za-z]?)"
        r"(?:\s*,?\s*(?:numeral|num\.?)\s*[0-9]+[A-Za-z]?)?"
        r"\s+(?:del|de la|de el)\s+"
        r"(?P<fuente>.+?)\s*$",
        texto,
        flags=re.IGNORECASE,
    )
    if match:
        return (
            normalizar_fuente(match.group("fuente")),
            normalizar_llave(match.group("numero")),
        )

    # Norma sin artículo específico: no es error.
    return None, None


pares[["fuente_key", "numero_key"]] = pd.DataFrame(
    pares["articulo"].map(parsear_articulo).tolist(),
    index=pares.index,
)

diccionario["fuente_key"] = diccionario["fuente"].map(normalizar_fuente)
diccionario["numero_key"] = diccionario["numero"].map(normalizar_llave)

menciona_articulo = pares["articulo"].astype(str).str.contains(
    r"\bArt(?:[íi]culo|\.)?",
    case=False,
    regex=True,
)

errores_reales = pares[pares["fuente_key"].isna() & menciona_articulo]

normas_sin_articulo = pares[pares["fuente_key"].isna() & ~menciona_articulo]

print("Normas completas sin artículo específico:", len(normas_sin_articulo))
display(normas_sin_articulo[["articulo", "sentencia_origen"]].drop_duplicates())
if len(errores_reales) > 0:
    print(
        " Hay Citas con artículo que no se pudieron interpretar. "
    )
    display(errores_reales[["articulo", "sentencia_origen"]].drop_duplicates())

df = pares.merge(
    diccionario[["fuente_key", "numero_key", "fuente", "texto_completo"]],
    on=["fuente_key", "numero_key"],
    how="left",
)

sin_texto = df["texto_completo"].fillna("").astype(str).str.strip().eq("")
print("Pares sin texto normativo completo:", sin_texto.sum())
print("\nDistribución antes de retirar faltantes:")
display(df["label"].value_counts().sort_index())

# Un cross-encoder necesita el texto del artículo; una cita corta no es suficiente.
df = df.loc[~sin_texto].copy().reset_index(drop=True)

# texto_completo ya trae el número de artículo en su encabezado, pero no la fuente —
# por eso se antepone acá.
df["texto_input"] = df["fuente"].astype(str) + ". " + df["texto_completo"].astype(str)
df["label"] = df["label"].astype(int)

columnas_modelo = [
    "consulta", "texto_input", "label",
    "sentencia_origen", "articulo", "tipo"
]
df = df[columnas_modelo]

print("\nPares utilizables:", len(df))
print("Sentencias únicas:", df["sentencia_origen"].nunique())
display(df["label"].value_counts().sort_index())
display(df.head(2))


Normas completas sin artículo específico: 21


,articulo,sentencia_origen
27,Ley 1636 de 2013,SU-075/18
111,Ordenanza 008 de 1986,T-282/15
117,Decreto Ley 2351 de 1965,T-1040/06
157,Ley 790 de 2002,T-866/05
158,Ley 813 de 2003,T-866/05
181,Decreto 190 de 2003,T-206/06
208,Ley 1822 de 2017,T-043/20
221,Ley 361 de 1997,T-198/06
233,Ley 361 de 1997,T-195/22
256,Ley 361 de 1997,T-076/24


Pares sin texto normativo completo: 23

Distribución antes de retirar faltantes:


,count
label,
0,276
1,285



Pares utilizables: 538
Sentencias únicas: 138


,count
label,
0,276
1,262


,consulta,texto_input,label,sentencia_origen,articulo,tipo
0,Trabajé como operador de bus articulado desde ...,"CST. ARTICULO 127. Modificado por el art. 14, ...",1,SL-3630/22,CST Art. 127,positivo
1,Trabajé como operador de bus articulado desde ...,"CST. ARTICULO 128. Modificado por el art. 15, ...",1,SL-3630/22,CST Art. 128,positivo


Hay que recordar que ya se conocía el hecho de que había 21  artículos que no se podían relacionar ya que pese a que existía una sentencia, faltaba dicho articulo para poder vincularlo a una descripción, por lo que estas filas se descartan para el entrenamiento del modelo

# 1.1 separación del dataset: Split train / validation / test sin fuga

In [5]:
from sklearn.model_selection import GroupShuffleSplit

# Primero separamos un test final: no se usa para escoger hiperparámetros.
split_test = GroupShuffleSplit(
    n_splits=1,
    test_size=0.15,
    random_state=SEED,
)

# Agrupamos por sentencia_origen: filas de una misma sentencia comparten consulta —
# dividir por fila filtraría la misma consulta a más de un split.
idx_train_val, idx_test = next(
    split_test.split(df, groups=df["sentencia_origen"])
)

train_val_df = df.iloc[idx_train_val].reset_index(drop=True)
test_df = df.iloc[idx_test].reset_index(drop=True)

# Del 85% restante, 15/85 equivale a 17.65%; deja ~70/15/15 total.
split_val = GroupShuffleSplit(
    n_splits=1,
    test_size=0.15 / 0.85,
    random_state=SEED,
)

idx_train, idx_val = next(
    split_val.split(train_val_df, groups=train_val_df["sentencia_origen"])
)

train_df = train_val_df.iloc[idx_train].reset_index(drop=True)
val_df = train_val_df.iloc[idx_val].reset_index(drop=True)

def resumen_split(nombre, datos):
    print(
        f"{nombre:10} | pares={len(datos):3} | "
        f"sentencias={datos['sentencia_origen'].nunique():3} | "
        f"positivos={datos['label'].sum():3} | "
        f"proporción positiva={datos['label'].mean():.3f}"
    )

resumen_split("train", train_df)
resumen_split("validation", val_df)
resumen_split("test", test_df)

# Verificación: ninguna sentencia debería estar en más de un conjunto.
grupos_train = set(train_df["sentencia_origen"])
grupos_val = set(val_df["sentencia_origen"])
grupos_test = set(test_df["sentencia_origen"])

sin_fuga = (
    grupos_train.isdisjoint(grupos_val)
    and grupos_train.isdisjoint(grupos_test)
    and grupos_val.isdisjoint(grupos_test)
)

print(f"\n{'✅' if sin_fuga else '⚠️'} Split agrupado, sin fuga entre sentencias: {sin_fuga}")

train      | pares=375 | sentencias= 96 | positivos=183 | proporción positiva=0.488
validation | pares= 87 | sentencias= 21 | positivos= 45 | proporción positiva=0.517
test       | pares= 76 | sentencias= 21 | positivos= 34 | proporción positiva=0.447

✅ Split agrupado, sin fuga entre sentencias: True


## 2. Modelo base + tokenizer, y tokenización

Cargamos el tokenizador de BETO y tokenizamos el dataset

In [6]:
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

MODELO = "dccuchile/bert-base-spanish-wwm-cased"
MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODELO)

def tokenizar(batch):
    # truncation="only_second": BETO tiene tope de 512 tokens; algunos artículos superan
    # 1800, la consulta nunca pasa de ~135 — se recorta solo el artículo.
    return tokenizer(
        batch["consulta"],
        batch["texto_input"],
        truncation="only_second",
        max_length=MAX_LENGTH,
    )

# Conservamos los DataFrames originales: los necesitaremos para ranking y ejemplos.
train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds = Dataset.from_pandas(val_df, preserve_index=False)
test_ds = Dataset.from_pandas(test_df, preserve_index=False)  # held-out: no se toca hasta la sección 6

train_tok = train_ds.map(tokenizar, batched=True)
val_tok = val_ds.map(tokenizar, batched=True)
test_tok = test_ds.map(tokenizar, batched=True)

# Dejamos únicamente lo que el modelo necesita para entrenar.
columnas_modelo = {"input_ids", "attention_mask", "token_type_ids", "label"}

def quitar_metadatos(dataset):
    columnas_a_quitar = [
        col for col in dataset.column_names
        if col not in columnas_modelo
    ]
    return dataset.remove_columns(columnas_a_quitar)

train_tok = quitar_metadatos(train_tok)
val_tok = quitar_metadatos(val_tok)
test_tok = quitar_metadatos(test_tok)

collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

print(train_tok)
print(val_tok)
print(test_tok)

# .keys() solo muestra los campos, no el contenido — se decodifica input_ids para verlo:
print("\nEjemplo tokenizado (primeras 20 subpalabras de input_ids):")
print(tokenizer.convert_ids_to_tokens(train_tok[0]["input_ids"])[:20])

config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/242k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/480k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

Map:   0%|          | 0/375 [00:00<?, ? examples/s]

Map:   0%|          | 0/87 [00:00<?, ? examples/s]

Map:   0%|          | 0/76 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 375
})
Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 87
})
Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 76
})

Ejemplo tokenizado (primeras 20 subpalabras de input_ids):
['[CLS]', 'Trabaj', '##é', 'como', 'operador', 'de', 'bus', 'articul', '##ado', 'desde', 'septiembre', 'de', '2004', 'hasta', 'septiembre', 'de', '2014', '.', 'Me', 'paga']


## 3. Baselines (contra qué comparamos)

Dos referencias: **clase mayoritaria** (el piso) y **BETO sin afinar** (mismo encoder,
cabeza recién inicializada). La comparación principal es sobre `validation`; la sección 6
confirma aparte que se sostiene también sobre `test`.

In [7]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def metricas_desde_predicciones(predicciones, labels):
    """Calcula métricas binarias cuando ya se tienen las clases predichas (no logits)."""
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predicciones,
        average="binary",
        zero_division=0,
    )

    return {
        "accuracy": accuracy_score(labels, predicciones),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

def metricas_clasificacion(logits, labels):
    """Convierte logits en clases (argmax) y reutiliza el cálculo común de métricas."""
    predicciones = np.argmax(logits, axis=-1)
    return metricas_desde_predicciones(predicciones, labels)

def cargar_modelo_base():
    # Fija también la inicialización de la cabeza clasificadora.
    fijar_semilla(SEED)

    return AutoModelForSequenceClassification.from_pretrained(
        MODELO,
        num_labels=2,
    )

In [8]:
# Baseline 1: predecir siempre la clase más frecuente en train (sin mirar val/test).
clase_mayoritaria = int(train_df["label"].value_counts().idxmax())

pred_mayoritaria_val = np.full(len(val_df), clase_mayoritaria, dtype=int)
pred_mayoritaria_test = np.full(len(test_df), clase_mayoritaria, dtype=int)

metricas_mayoritaria_val = metricas_desde_predicciones(
    pred_mayoritaria_val, val_df["label"].to_numpy()
)
metricas_mayoritaria_test = metricas_desde_predicciones(
    pred_mayoritaria_test, test_df["label"].to_numpy()
)

nombre_clase = "RELEVANTE" if clase_mayoritaria == 1 else "NO RELEVANTE"
print(f"Clase mayoritaria en train: {clase_mayoritaria} ({nombre_clase})")
print("\n=== BASELINE 1: CLASE MAYORITARIA (validation) ===")
for nombre, valor in metricas_mayoritaria_val.items():
    print(f"{nombre:10}: {valor:.3f}")

Clase mayoritaria en train: 0 (NO RELEVANTE)

=== BASELINE 1: CLASE MAYORITARIA (validation) ===
accuracy  : 0.483
precision : 0.000
recall    : 0.000
f1        : 0.000


In [9]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# Baseline 2 (el recomendado por la asignación): el mismo BETO sin fine-tuning,
# cabeza clasificadora recién inicializada.
modelo_baseline = cargar_modelo_base().to(device)

args_baseline = TrainingArguments(
    output_dir="./baseline_temporal",
    per_device_eval_batch_size=16,
    report_to="none",
    seed=SEED,
)

trainer_baseline = Trainer(
    model=modelo_baseline,
    args=args_baseline,
    processing_class=tokenizer,
    data_collator=collator,
)

pred_baseline = trainer_baseline.predict(val_tok)
pred_baseline_test = trainer_baseline.predict(test_tok)

metricas_baseline_val = metricas_clasificacion(
    pred_baseline.predictions,
    pred_baseline.label_ids,
)
metricas_baseline_test = metricas_clasificacion(
    pred_baseline_test.predictions,
    pred_baseline_test.label_ids,
)

print("=== BASELINE 2: BETO SIN FINE-TUNING (validation) ===")
for nombre, valor in metricas_baseline_val.items():
    print(f"{nombre:10}: {valor:.3f}")

# Liberamos GPU antes de cargar el modelo LoRA.
del modelo_baseline, trainer_baseline
torch.cuda.empty_cache()

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  440MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

=== BASELINE 2: BETO SIN FINE-TUNING (validation) ===
accuracy  : 0.483
precision : 0.000
recall    : 0.000
f1        : 0.000


## 4. LoRA: fine-tuning eficiente


- `r` (rank): tamaño de las matrices pequeñas.
- `lora_alpha`: cuánto pesa el ajuste (regla común: `alpha ≈ 2×r`).
- `target_modules`: a qué capas se aplica. En BETO (BERT estándar) son `query` y `value`
  — en DistilBERT (el modelo del notebook base) son `q_lin` y `v_lin`.

In [10]:
from peft import LoraConfig, TaskType, get_peft_model

modelo_lora = cargar_modelo_base()

lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value"],  # BETO usa query/value, no q_lin/v_lin (ver nota arriba)
    modules_to_save=["classifier"],
)

modelo_lora = get_peft_model(modelo_lora, lora_cfg)
modelo_lora.to(device)

modelo_lora.print_trainable_parameters()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	

trainable params: 296,450 || all params: 110,148,868 || trainable%: 0.2691


## 5. Entrenar con la Trainer API

La `Trainer` nos regala el loop de entrenamiento. Le damos el modelo, los datos, la métrica y unos
hiperparámetros mínimos. Entrena hasta 8 épocas con early stopping — con LoRA eso basta para ver
el salto sobre el baseline.

In [11]:
import math

from transformers import EarlyStoppingCallback

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return metricas_clasificacion(logits, labels)

BATCH_SIZE = 8
GRAD_ACCUM = 2
EPOCHS = 8

# warmup_steps calculado (no fijo) para que siga siendo ~10% del entrenamiento aunque
# cambie el tamaño de train; evita también el warning de warmup_ratio deprecado.
pasos_por_epoca = math.ceil(math.ceil(len(train_tok) / BATCH_SIZE) / GRAD_ACCUM)
total_pasos = pasos_por_epoca * EPOCHS
warmup_steps = round(0.10 * total_pasos)

args_lora = TrainingArguments(
    output_dir="./lawten_lora_out",
    learning_rate=1e-4,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=GRAD_ACCUM,  # batch efectivo = 16
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    warmup_steps=warmup_steps,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    logging_strategy="steps",
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    optim="adamw_torch",

    seed=SEED,
    data_seed=SEED,
    report_to="none",
)

trainer = Trainer(
    model=modelo_lora,
    args=args_lora,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=2)
    ],
)

resultado_entrenamiento = trainer.train()

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.385260,0.701626,0.540230,0.777778,0.155556,0.259259
2,1.354993,0.643597,0.678161,0.904762,0.422222,0.575758
3,1.140225,0.542997,0.781609,0.933333,0.622222,0.746667
4,0.987453,0.491515,0.793103,0.935484,0.644444,0.763158
5,0.728045,0.452318,0.793103,0.935484,0.644444,0.763158
6,0.826902,0.464547,0.793103,0.935484,0.644444,0.763158


## 6. Evaluar y comparar contra los baselines

Comparación principal sobre `validation` (lo que pide la asignación): clase mayoritaria,
BETO sin afinar y BETO + LoRA juntos. Al final, una confirmación aparte sobre `test` — un
conjunto que el modelo no tocó ni para entrenar ni para elegir checkpoint.

In [12]:
metricas_lora_val = trainer.evaluate()
metricas_lora_val_limpias = {
    nombre: metricas_lora_val[f"eval_{nombre}"]
    for nombre in ["accuracy", "precision", "recall", "f1"]
}

def fila_metricas(nombre_modelo, metricas):
    return {
        "modelo": nombre_modelo,
        "accuracy": metricas["accuracy"],
        "precision": metricas["precision"],
        "recall": metricas["recall"],
        "f1": metricas["f1"],
    }

tabla_clasificacion_val = pd.DataFrame([
    fila_metricas("Clase mayoritaria", metricas_mayoritaria_val),
    fila_metricas("BETO sin afinar (zero-shot)", metricas_baseline_val),
    fila_metricas("BETO + LoRA", metricas_lora_val_limpias),
]).set_index("modelo")

print("=== COMPARACIÓN PRINCIPAL EN VALIDATION ===")
display(tabla_clasificacion_val.round(3))

delta_clasificacion_val = (
    tabla_clasificacion_val.loc["BETO + LoRA"]
    - tabla_clasificacion_val.loc["BETO sin afinar (zero-shot)"]
)
print("\nMejora de LoRA sobre BETO sin afinar:")
display(delta_clasificacion_val.round(3).rename("delta").to_frame().T)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.826902,0.491515,6,0.793103,0.935484,0.644444,0.763158


=== COMPARACIÓN PRINCIPAL EN VALIDATION ===


,accuracy,precision,recall,f1
modelo,,,,
Clase mayoritaria,0.483,0.000,0.000,0.000
BETO sin afinar (zero-shot),0.483,0.000,0.000,0.000
BETO + LoRA,0.793,0.935,0.644,0.763



Mejora de LoRA sobre BETO sin afinar:


,accuracy,precision,recall,f1
delta,0.31,0.935,0.644,0.763


### Confirmación sobre test (no reemplaza la comparación anterior)

`test` nunca se usó para entrenar ni para elegir el checkpoint — sirve para chequear que la
mejora no es un efecto de haber ajustado la selección de época a `validation`.

In [13]:
metricas_lora_test = trainer.evaluate(eval_dataset=test_tok)
metricas_lora_test_limpias = {
    nombre: metricas_lora_test[f"eval_{nombre}"]
    for nombre in ["accuracy", "precision", "recall", "f1"]
}

tabla_clasificacion_test = pd.DataFrame([
    fila_metricas("Clase mayoritaria", metricas_mayoritaria_test),
    fila_metricas("BETO sin afinar (zero-shot)", metricas_baseline_test),
    fila_metricas("BETO + LoRA", metricas_lora_test_limpias),
]).set_index("modelo")

print("=== CONFIRMACIÓN EN TEST (held-out) ===")
display(tabla_clasificacion_test.round(3))

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.826902,0.346540,6,0.894737,0.906250,0.852941,0.878788


=== CONFIRMACIÓN EN TEST (held-out) ===


,accuracy,precision,recall,f1
modelo,,,,
Clase mayoritaria,0.553,0.000,0.000,0.000
BETO sin afinar (zero-shot),0.553,0.000,0.000,0.000
BETO + LoRA,0.895,0.906,0.853,0.879


## 7. Ranking por sentencia: Recall@k y Precision@k

La sección 6 evalúa fila por fila; el producto reordena candidatos *por sentencia* — importa
si los artículos correctos quedan en el top-k. Comparación con `k=3` sobre `validation`.
Clase mayoritaria queda `N/A`: mismo score para todos no define un ranking real.

In [14]:
def probabilidad_positiva(logits):
    logits = np.asarray(logits)
    exp_logits = np.exp(logits - logits.max(axis=1, keepdims=True))
    probabilidades = exp_logits / exp_logits.sum(axis=1, keepdims=True)
    return probabilidades[:, 1]

def metricas_ranking(datos, logits, k=3):
    """Ordena por score dentro de cada sentencia y calcula Recall@k/Precision@k promediados."""
    evaluacion = datos.copy().reset_index(drop=True)
    evaluacion["score_relevancia"] = probabilidad_positiva(logits)

    resultados = []

    for sentencia, grupo in evaluacion.groupby("sentencia_origen"):
        ranking = grupo.sort_values("score_relevancia", ascending=False)
        top_k = ranking.head(k)

        positivos_totales = grupo["label"].sum()
        positivos_en_top_k = top_k["label"].sum()

        # Sentencia sin positivos en este split (no resolvió en el join): recall queda NaN.
        recall_at_k = (
            positivos_en_top_k / positivos_totales if positivos_totales > 0 else np.nan
        )

        resultados.append({
            "sentencia_origen": sentencia,
            "positivos_totales": positivos_totales,
            "positivos_en_top_k": positivos_en_top_k,
            "recall_at_k": recall_at_k,
            "precision_at_k": positivos_en_top_k / len(top_k),
        })

    resultados = pd.DataFrame(resultados)

    return {
        f"recall@{k}": resultados["recall_at_k"].mean(),
        f"precision@{k}": resultados["precision_at_k"].mean(),
        "detalle_por_sentencia": resultados,
    }

In [15]:
K_PRINCIPAL = 3

pred_lora_val = trainer.predict(val_tok)

ranking_baseline_val = metricas_ranking(val_df, pred_baseline.predictions, k=K_PRINCIPAL)
ranking_lora_val = metricas_ranking(val_df, pred_lora_val.predictions, k=K_PRINCIPAL)

col_recall = f"recall@{K_PRINCIPAL}"
col_precision = f"precision@{K_PRINCIPAL}"

tabla_ranking_val = pd.DataFrame([
    {"modelo": "Clase mayoritaria", col_recall: np.nan, col_precision: np.nan},
    {
        "modelo": "BETO sin afinar (zero-shot)",
        col_recall: ranking_baseline_val[col_recall],
        col_precision: ranking_baseline_val[col_precision],
    },
    {
        "modelo": "BETO + LoRA",
        col_recall: ranking_lora_val[col_recall],
        col_precision: ranking_lora_val[col_precision],
    },
]).set_index("modelo")

print(f"=== RANKING PRINCIPAL EN VALIDATION: k={K_PRINCIPAL} ===")
display(tabla_ranking_val.round(3))

delta_recall = ranking_lora_val[col_recall] - ranking_baseline_val[col_recall]
delta_precision = ranking_lora_val[col_precision] - ranking_baseline_val[col_precision]
print(
    f"\nMejora de LoRA sobre BETO sin afinar: "
    f"{col_recall}={delta_recall:+.3f}, {col_precision}={delta_precision:+.3f}"
)
print("Clase mayoritaria: N/A porque todos sus candidatos empatan en score, no define un ranking real.")

=== RANKING PRINCIPAL EN VALIDATION: k=3 ===


,recall@3,precision@3
modelo,,
Clase mayoritaria,NaN,NaN
BETO sin afinar (zero-shot),0.679,0.381
BETO + LoRA,0.898,0.587



Mejora de LoRA sobre BETO sin afinar: recall@3=+0.219, precision@3=+0.206
Clase mayoritaria: N/A porque todos sus candidatos empatan en score, no define un ranking real.


In [16]:
detalle_val = ranking_lora_val["detalle_por_sentencia"]

display(
    detalle_val.sort_values("recall_at_k")
    .head(10)
)

,sentencia_origen,positivos_totales,positivos_en_top_k,recall_at_k,precision_at_k
3,SL-2858/22,5,3,0.600000,1.000000
4,SL-780/23,5,3,0.600000,1.000000
2,SL-2850/20,3,2,0.666667,0.666667
1,SL-1514/23,3,2,0.666667,0.666667
6,T-092/16,3,2,0.666667,0.666667
13,T-347/24,3,2,0.666667,0.666667
5,SU-428/23,3,3,1.000000,1.000000
0,SL-1050/23,1,1,1.000000,0.333333
7,T-1128/00,2,2,1.000000,0.666667
8,T-1136/00,1,1,1.000000,0.333333


### Confirmación sobre test (no reemplaza la comparación anterior)

In [17]:
pred_lora_test = trainer.predict(test_tok)

ranking_baseline_test = metricas_ranking(test_df, pred_baseline_test.predictions, k=K_PRINCIPAL)
ranking_lora_test = metricas_ranking(test_df, pred_lora_test.predictions, k=K_PRINCIPAL)

tabla_ranking_test = pd.DataFrame([
    {
        "modelo": "BETO sin afinar (zero-shot)",
        col_recall: ranking_baseline_test[col_recall],
        col_precision: ranking_baseline_test[col_precision],
    },
    {
        "modelo": "BETO + LoRA",
        col_recall: ranking_lora_test[col_recall],
        col_precision: ranking_lora_test[col_precision],
    },
]).set_index("modelo")

print(f"=== CONFIRMACIÓN EN TEST (held-out): k={K_PRINCIPAL} ===")
display(tabla_ranking_test.round(3))

=== CONFIRMACIÓN EN TEST (held-out): k=3 ===


,recall@3,precision@3
modelo,,
BETO sin afinar (zero-shot),0.729,0.333
BETO + LoRA,0.988,0.524


## 8. Ejemplos cualitativos (qué hace el modelo)

M1 pide al menos 3 ejemplos de entrada → salida. La selección sigue reglas explícitas (no al
azar): un verdadero positivo representativo, un falso negativo cerca del umbral, un negativo
difícil bien rechazado y, si existe, el falso positivo con mayor score — el error más costoso
para un asistente legal. El valor mostrado es un **score de relevancia**, no una confianza
calibrada; todos los casos vienen de `val_df`, con datos reales del dataset.

In [18]:
def formatear_snippet(texto, max_chars=200):
    """Recorta un texto largo a un snippet legible sin cortar palabras a la mitad."""
    texto = " ".join(str(texto).split())
    if len(texto) <= max_chars:
        return texto
    return texto[:max_chars].rsplit(" ", 1)[0] + "..."

def mostrar_ejemplo(numero, fila):
    """Muestra una fila real de val_df/test_df junto a la predicción del modelo y el label real."""
    # Reutiliza el score ya calculado si la fila viene de val_scored (evita recorrer el modelo).
    if "score_relevancia" in fila.index:
        score = float(fila["score_relevancia"])
    else:
        entrada = tokenizer(
            fila["consulta"],
            fila["texto_input"],
            return_tensors="pt",
            truncation="only_second",
            max_length=512,
        ).to(device)

        trainer.model.eval()
        with torch.no_grad():
            logits = trainer.model(**entrada).logits
            score = torch.softmax(logits, dim=-1)[0, 1].item()

    prediccion = "RELEVANTE" if score >= 0.5 else "NO RELEVANTE"
    esperado = "RELEVANTE" if fila["label"] == 1 else "NO RELEVANTE"
    acierto = "✅ acierta" if prediccion == esperado else "❌ falla"
    caso = fila.get("caso_cualitativo", "Caso cualitativo")

    print(f"Ejemplo {numero} — {caso}")
    print(f"  Tipo     : {fila['tipo']}, sentencia: {fila['sentencia_origen']}")
    print(f"  Consulta : {formatear_snippet(fila['consulta'], 300)}")
    print(f"  Artículo : {fila['articulo']}")
    print(f"             {formatear_snippet(fila['texto_input'], 200)}")
    print(f"  Esperado : {esperado}")
    print(f"  Predicho : {prediccion} (score de relevancia={score:.3f})  {acierto}")
    print()

In [19]:
# Ejemplos reales de validation, seleccionados con criterios explícitos (no al azar):
val_scored = val_df.copy()
val_scored["score_relevancia"] = probabilidad_positiva(pred_lora_val.predictions)
val_scored["prediccion"] = (val_scored["score_relevancia"] >= 0.5).astype(int)

# 1. Verdadero positivo representativo: el más cercano a la mediana de score (no el más fácil).
verdaderos_positivos = val_scored[
    (val_scored["label"] == 1) & (val_scored["prediccion"] == 1)
].copy()
mediana_vp = verdaderos_positivos["score_relevancia"].median()
positivo_representativo = (
    verdaderos_positivos
    .assign(distancia_mediana=lambda d: (d["score_relevancia"] - mediana_vp).abs())
    .sort_values(["distancia_mediana", "sentencia_origen", "articulo"])
    .head(1)
    .drop(columns="distancia_mediana")
)
positivo_representativo["caso_cualitativo"] = "Verdadero positivo representativo"

# 2. Falso negativo más cercano al umbral.
falso_negativo = (
    val_scored[(val_scored["label"] == 1) & (val_scored["prediccion"] == 0)]
    .sort_values("score_relevancia", ascending=False)
    .head(1)
)
falso_negativo["caso_cualitativo"] = "Falso negativo más cercano al umbral"

# 3. Negativo difícil correctamente rechazado.
negativo_dificil_correcto = (
    val_scored[
        (val_scored["tipo"] == "negativo_dificil_placeholder")
        & (val_scored["label"] == 0)
        & (val_scored["prediccion"] == 0)
    ]
    .sort_values("score_relevancia", ascending=False)
    .head(1)
)
negativo_dificil_correcto["caso_cualitativo"] = "Negativo difícil correctamente rechazado"

# 4. Falso positivo con mayor score, si existe (ver nota arriba sobre por qué importa).
falso_positivo = (
    val_scored[(val_scored["label"] == 0) & (val_scored["prediccion"] == 1)]
    .sort_values("score_relevancia", ascending=False)
    .head(1)
)
if not falso_positivo.empty:
    falso_positivo["caso_cualitativo"] = "Falso positivo con mayor score"

selecciones = [
    caso for caso in [
        positivo_representativo,
        falso_negativo,
        negativo_dificil_correcto,
        falso_positivo,
    ]
    if not caso.empty
]

ejemplos_cualitativos = pd.concat(selecciones).reset_index(drop=True)

for numero, (_, fila) in enumerate(ejemplos_cualitativos.iterrows(), start=1):
    mostrar_ejemplo(numero, fila)

Ejemplo 1 — Verdadero positivo representativo
  Tipo     : positivo, sentencia: SL-2850/20
  Consulta : Trabajé 24 años como operario de montacargas y me despidieron después de que un compañero ebrio me agredió verbalmente con insultos racistas llamándome 'negro hijueputa' y 'negro bruto'. Yo reaccioné lanzándole mi casco. La empresa nos despidió a los dos por igual, sin considerar que yo solo me...
  Artículo : Ley 50 de 1990, Art. 6
             Ley 50 de 1990. artículo 6o del Código Sustantivo del Trabajo. 2. Cuando se requiere reemplazar personal en vacaciones, en uso de licencia, en incapacidad por enfermedad o maternidad. 3. Para atender...
  Esperado : RELEVANTE
  Predicho : RELEVANTE (score de relevancia=0.795)  ✅ acierta

Ejemplo 2 — Falso negativo más cercano al umbral
  Tipo     : positivo, sentencia: SU-428/23
  Consulta : Trabajé como odontóloga desde el año 2000. Sufrí dos accidentes laborales que me causaron una enfermedad profesional en la mano derecha (Tenosinovitis). 

## 9. Guardar el adaptador LoRA (opcional)

LoRA solo guarda las matrices pequeñas: son unos pocos MB, no el modelo entero.

In [20]:
modelo_lora.save_pretrained('./s04_lora_adapter')
print('Adaptador LoRA guardado en ./s04_lora_adapter (solo los pesos de LoRA).')

Adaptador LoRA guardado en ./s04_lora_adapter (solo los pesos de LoRA).
